# 05 – Odszumianie nagrań: trening modeli

<div style="border-left: 4px solid #2b6cb0; padding: 12px 16px; margin-bottom: 20px; border-radius: 0 4px 4px 0;">
  <strong>Praca Inżynierska:</strong> Analiza i klasyfikacja dźwięków z wykorzystaniem uczenia maszynowego <br>
  <strong>Autorzy:</strong> Norbert Gwiazda, Mateusz Ciurzyński | <strong>Data:</strong> 2026-11-22<br>
  <strong>Zbiory danych:</strong> Respiratory Sound Database (ICBHI 2017), HF_Lung_V2, SPRSound<br>
  <strong>Cel:</strong> Trening i porównanie sieci odszumiających nagrania oddechowe
</div>

---

## Spis treści
- [1. Wprowadzenie](#1-wprowadzenie)
- [2. Środowisko i import bibliotek](#2-środowisko-i-import-bibliotek)
- [3. Konfiguracja](#3-konfiguracja)
- [4. Dane treningowe](#4-dane-treningowe)
  - [4.1. Wybór nagrań](#41-wybór-nagrań)
  - [4.2. Selekcja czystych nagrań](#42-selekcja-czystych-nagrań)
  - [4.3. Pary czysty / zaszumiony sygnał](#43-pary-czysty--zaszumiony-sygnał)
- [5. Model i przepis treningowy](#5-model-i-przepis-treningowy)
- [6. Trening](#6-trening)
- [7. Ocena](#7-ocena)
  - [7.1. Krzywe uczenia](#71-krzywe-uczenia)
  - [7.2. Jakość według poziomu szumu](#72-jakość-według-poziomu-szumu)
  - [7.3. Przykłady odszumiania](#73-przykłady-odszumiania)
- [8. Porównanie modeli](#8-porównanie-modeli)
- [9. Użycie w pipeline](#9-użycie-w-pipeline)

---

## 1. Wprowadzenie

Nagrania osłuchowe zawierają szumy niezwiązane z oddychaniem: tarcie membrany stetoskopu o skórę, dźwięki otoczenia szpitalnego (szum urządzeń, sygnały monitorów, rozmowy) oraz odgłosy samego pacjenta (kaszel, mowa, ruchy). Ten notatnik trenuje sieci, które usuwają takie zakłócenia z sygnału, zanim trafi on do ekstrakcji cech i klasyfikatora. Ta sama pętla treningowa obsługuje wszystkie modele; model wybiera się jedną zmienną w sekcji 3.

Podejście jest wzorowane na pracy Tzeng i in. (JMIR AI, 2025), w której porównano cztery architektury jako moduł odszumiający przed klasyfikacją dźwięków oddechowych:

| Model | Dziedzina | Parametry* | Status w projekcie |
|---|---|---|---|
| Wave-U-Net (Stoller i in., 2018) | fala | 3,3 M | zaimplementowany |
| MANNER (Park i in., 2022) | fala | 24 M | zaimplementowany |
| PHASEN (Yin i in., 2020) | widmo | 7,7 M | planowany |
| CMGAN (Cao i in., 2022) | widmo | 1,8 M | planowany |

<small>* liczby parametrów według Tzeng i in.; konfiguracje w tym projekcie mogą się różnić (liczba jest wypisywana w sekcji 5).</small>

Wspólne założenia dla wszystkich modeli:

* do nagrań dodawany jest szum o zadanym stosunku sygnału do szumu (SNR): **15/10/5/0 dB** w treningu oraz **17,5/12,5/7,5/2,5 dB** z niewidzianym szumem w walidacji,
* sieć uczy się odtwarzać sygnał sprzed dodania szumu, na **segmentach 4 s** z losowym przesunięciem w czasie,
* **brak wycieku danych** – używane są wyłącznie nagrania z części treningowej klasyfikatora (`split=0.7`, `random_state=42`, jak w `MultiDatasetPipelineBuilder`); walidacja to 10% pacjentów odłożonych z tej części,
* **zgodność treningu z inferencją** – nagrania są wczytywane tą samą funkcją co w pipeline (4 kHz), szum jest mieszany przy 4 kHz, a sygnał przechodzi przez sieć tak samo jak przy odszumianiu,
* **szum syntetyczny** – zbiór NTUH Clinical Noise z pracy Tzeng i in. nie jest publiczny, dlatego szum jest generowany proceduralnie w tych samych trzech kategoriach (tarcie, otoczenie, pacjent); szum walidacyjny ma inne parametry i inne źródło losowości,
* **prawdziwe zakłócenia** – opcjonalnie część próbek szumu pochodzi z korpusu MUSAN (mowa, muzyka, szumy; Snyder i in., 2015), a druga walidacja mierzy jakość na prawdziwym szumie od plików nieużytych w treningu.

Elementy zależne od modelu – funkcja straty, optymalizator, harmonogram współczynnika uczenia oraz domyślny rozmiar batcha i liczba epok – są zebrane w **przepisach treningowych** (`src/denoising/training.py`).

---

## 2. Środowisko i import bibliotek

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'
%config InlineBackend.close_figures = True

import sys
import time
import warnings
from contextlib import nullcontext
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Audio, display
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Skrypty lokalne
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data_pipeline import _load_feature_audio
from src.denoising import (
    MODELS,
    build_model,
    enhance_signal,
    load_model,
    load_weights,
    model_config,
)
from src.denoising.audio_tagging import (
    RESPIRATORY_LABELS,
    TZENG_EXCLUDED_LABELS,
    load_tagger_audio,
    select_clean,
    tag_recordings,
)
from src.denoising.base import load_checkpoint
from src.denoising.musan import MusanNoise, build_musan_cache, list_musan, split_musan
from src.denoising.noise import (
    NOISE_VERSION,
    TEST_SNRS_DB,
    TRAIN_SNRS_DB,
    FileNoiseBank,
    MixedNoise,
    SyntheticClinicalNoise,
)
from src.denoising.training import recipe_for
from src.denoising.training_data import (
    PIPELINE_SAMPLE_RATE,
    NoisyCleanDataset,
    build_cache,
    collect_train_recordings,
    si_sdr,
    split_by_group,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}"
      + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print("Dostępne modele:", ", ".join(MODELS))

## 3. Konfiguracja

**`MODEL`** wybiera sieć. Parametry architektury każdego modelu są w `MODEL_OPTIONS`, a ustawienia treningu w `TRAIN` – wartość `None` oznacza domyślną wartość z przepisu danego modelu (wypisywaną w sekcji 5).

* `model_sample_rate = 4000` – pipeline podaje sygnał 4 kHz (pasmo do 2 kHz), więc wyższa częstotliwość próbkowania nie wnosi informacji, a zwiększa koszt obliczeń.
* `INIT_CHECKPOINT` – wagi startowe do dostrajania, np. oficjalne wagi MANNER wytrenowane na mowie (`manner_base.pth`, 16 kHz; wtedy `model_sample_rate=16000`). Tzeng i in. dostrajali takie wagi przez 20 epok z `lr = 5e-5`; dla MANNER ustaw wtedy też `"one_cycle": False` w `RECIPE_OPTIONS`.
* `RUN_NAME` – nazwa folderu z wynikami w `models/`. Notatnik **nie nadpisze** istniejącego treningu: jeśli w folderze jest `best.pt`, trzeba ustawić `resume = True` (kontynuacja) albo `OVERWRITE = True`.
* `SMOKE_TEST = True` uruchamia krótki przebieg (tylko ICBHI, kilka kroków), który pozwala sprawdzić, czy cały notatnik działa.

In [ ]:
MODEL = "cmgan"  # "wave-u-net" | "manner" | "cmgan"
SMOKE_TEST = False

MODEL_OPTIONS = {
    "wave-u-net": {
        "model_sample_rate": 4000,
        "num_layers": 12,
        "num_filters": 24,
        "down_kernel_size": 15,
        "up_kernel_size": 5,
        "peak_normalize": True,
    },
    "manner": {
        "model_sample_rate": 4000,
        "variant": "base",  # "small" | "base" | "large"
    },
    'cmgan': {"model_sample_rate": 4000, "n_fft": 100, "hop_length": 25}
}
RECIPE_OPTIONS = {
    "wave-u-net": {"stft_loss_weight": 0.0},
    "manner": {"one_cycle": True},
    "cmgan": {"discriminator": True, "cut_seconds": 2.0}
}

TRAIN = {
    "epochs": 50,          # None = domyślna wartość z przepisu modelu
    "batch_size": 8,
    "lr": None,
    "grad_clip": 0.0,        # 0 = bez przycinania gradientu
    "patience": 0,           # early stopping, 0 = wyłączony
    "max_steps": None,       # limit kroków na epokę
    "num_workers": 4,
    "amp": True,             # bfloat16 na GPU
    "seed": 0,
    "resume": True,
}
DATA = {
    "datasets": ("spr", "icbhi", "hf"),
    "split": 0.7,            # podział train/test klasyfikatora (jak w pipeline)
    "random_state": 42,
    "val_fraction": 0.1,     # odsetek pacjentów odłożonych do walidacji
    "val_segments": 512,
    "segment_seconds": 4.0,
}

INIT_CHECKPOINT = None   # np. ROOT / "weights" / "manner_base.pth"
NOISE_DIR = None         # folder z nagraniami szumu; None = szum syntetyczny

# Prawdziwe zakłócenia z korpusu MUSAN (mowa, muzyka, szumy), sekcja 4.3
MUSAN = {
    "enabled": True,
    "root": ROOT / "data" / "musan",
    "weight": 0.5,  # udział MUSAN wśród próbek szumu treningowego (reszta: szum syntetyczny)
    "category_weights": {"speech": 0.5, "music": 0.2, "noise": 0.3},
    "max_hours": {"speech": 10.0, "music": 5.0, "noise": None},  # None = wszystkie pliki
    "coupling_cutoff_hz": (500.0, 1500.0),  # filtr „stetoskopowy”; None = bez filtra
    "exclude_respiratory": True,  # pomiń szumy MUSAN rozpoznane przez CNN14 jako oddech
}

# Selekcja czystych nagrań treningowych taggerem CNN14 (Tzeng i in., 2025), sekcja 4.2
CLEAN_SELECTION = {
    "enabled": True,
    "labels": TZENG_EXCLUDED_LABELS,  # etykiety AudioSet wykluczające nagranie
    "threshold": 0.5,                  # prawdopodobieństwo, od którego etykieta wyklucza
    "weights": ROOT / "models" / "panns" / "Cnn14_mAP=0.431.pth",
    "labels_csv": ROOT / "models" / "panns" / "class_labels_indices.csv",
}
RUN_NAME = None          # None = nazwa modelu
OVERWRITE = False

if SMOKE_TEST:
    DATA.update(datasets=("icbhi",), val_segments=32)
    TRAIN.update(epochs=2, max_steps=15, num_workers=0)

RUN_NAME = (RUN_NAME or MODEL.replace("-", "_")) + ("_smoke" if SMOKE_TEST else "")
DATA_DIR = ROOT / "data"
CACHE_DIR = DATA_DIR / "cache" / "denoising_4k"
TAGS_CACHE = DATA_DIR / "cache" / "audio_tags_cnn14.parquet"
MUSAN_CACHE = DATA_DIR / "cache" / "musan_4k"
OUT_DIR = ROOT / "models" / RUN_NAME
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(TRAIN["seed"])

if (OUT_DIR / "best.pt").exists() and not (TRAIN["resume"] or OVERWRITE or SMOKE_TEST):
    raise FileExistsError(
        f"{OUT_DIR} zawiera już wytrenowany model. Ustaw TRAIN['resume'] = True, "
        "inną nazwę RUN_NAME albo OVERWRITE = True."
    )
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Model: {MODEL}, urządzenie: {DEVICE}, wyniki: {OUT_DIR}")

---

## 4. Dane treningowe

### 4.1. Wybór nagrań

`collect_train_recordings` odtwarza podział każdego zbioru na część treningową i testową klasyfikatora i zwraca **tylko nagrania treningowe**. Następnie `split_by_group` odkłada część pacjentów do walidacji modelu odszumiającego. HF Lung nie udostępnia identyfikatora pacjenta, więc tam walidacja jest wydzielana po nagraniach; wpływa to tylko na wybór najlepszej epoki, nie na ocenę klasyfikatora.

Przy pierwszym uruchomieniu wszystkie nagrania są jednorazowo przeliczane do 4 kHz i zapisywane jako `.npy` w `data/cache/denoising_4k/` (dla trzech zbiorów ok. 2,7 GB). Cache jest wspólny dla wszystkich modeli.

In [ ]:
recordings = collect_train_recordings(DATA_DIR, DATA["datasets"], DATA["split"], DATA["random_state"])
build_cache(recordings, CACHE_DIR)
train_recs, val_recs = split_by_group(recordings, DATA["val_fraction"], TRAIN["seed"])

summary = pd.DataFrame(
    [{"zbiór": r.source, "część": "trening", "grupa": r.group} for r in train_recs]
    + [{"zbiór": r.source, "część": "walidacja", "grupa": r.group} for r in val_recs]
)
display(
    summary.groupby(["zbiór", "część"])
    .agg(nagrania=("grupa", "size"), pacjenci_lub_nagrania=("grupa", "nunique"))
    .unstack("część")
)

### 4.2. Selekcja czystych nagrań

Nagrania ze zbiorów same zawierają zakłócenia (mowa personelu, płacz dziecka, szum urządzeń), a w parach treningowych pełnią rolę „czystego” wzorca. Tzeng i in. (2025) odrzucali więc nagrania, w których model rozpoznawania dźwięków **CNN14** (PANNs, Kong i in., 2020; AudioSet, 527 klas) wykrywał jedną z etykiet: *Music*, *Speech*, *Fire*, *Animal*, *Cat*, *Domestic animals, pets* – wybranych spośród jego najczęstszych predykcji na tych danych. Z 920 nagrań ICBHI pozostawili 889 (odrzucone 3,4%), a ręczna weryfikacja dała 92,5% trafności tagowania.

Szczegóły tej implementacji:

* **próg** – praca nie podaje progu prawdopodobieństwa, więc jest parametrem (`CLEAN_SELECTION["threshold"]`); tabela poniżej pokazuje odsetek odrzuceń dla kilku progów, co pozwala porównać go z 3,4% dla ICBHI z pracy,
* **wejście taggera** – oryginalne pliki przeliczone do 32 kHz (częstotliwość CNN14), a nie sygnał 4 kHz z pipeline, aby tagger dostał całe dostępne pasmo; nagrania zapisane w 4 kHz (HF Lung, część ICBHI) i 8 kHz (SPRSound) mają jednak pasmo tylko do 2 / 4 kHz, na którym CNN14 nie był uczony – decyzje warto sprawdzić odsłuchem (komórki niżej),
* **co jest filtrowane** – tylko część treningowa denoisera; walidacja zostaje bez zmian, żeby wyniki z selekcją i bez niej były porównywalne; klasyfikator nadal korzysta ze wszystkich nagrań,
* **kontrola proporcji klas** – jeżeli nagrania z trzeszczeniami lub świstami byłyby odrzucane częściej niż pozostałe, denoiser uczyłby się głównie na nagraniach bez objawów; tabela poniżej porównuje ich udział przed i po selekcji.

Wyniki tagowania są zapisywane w `data/cache/audio_tags_cnn14.parquet` (jednorazowo ok. 25 min dla wszystkich nagrań), więc zmiana progu lub listy etykiet nie wymaga ponownego uruchamiania sieci. Wagi CNN14 (`Cnn14_mAP=0.431.pth`, [Zenodo](https://zenodo.org/record/3987831)) i lista etykiet AudioSet (`class_labels_indices.csv`) muszą znajdować się w `models/panns/`.

In [ ]:
SELECTION_SUMMARY = None
if CLEAN_SELECTION["enabled"]:
    tags = tag_recordings(
        [r.path for r in recordings],
        TAGS_CACHE,
        weights=CLEAN_SELECTION["weights"],
        labels=CLEAN_SELECTION["labels_csv"],
        device=str(DEVICE),
    )
    source_of = {str(r.path): r.source for r in recordings}

    # Najczęstsze etykiety o najwyższym prawdopodobieństwie w każdym zbiorze
    top_label = tags.idxmax(axis=1).rename("etykieta").to_frame()
    top_label["zbiór"] = top_label.index.map(source_of)
    top_counts = (
        top_label.groupby("zbiór")["etykieta"].value_counts(normalize=True)
        .groupby(level=0).head(8).mul(100).round(1).rename("% nagrań")
    )
    display(top_counts.to_frame())
else:
    print("Selekcja wyłączona – trening na wszystkich nagraniach.")

In [ ]:
if CLEAN_SELECTION["enabled"]:
    train_paths = {str(r.path) for r in train_recs}
    train_tags = tags.loc[tags.index.isin(train_paths)]

    # Odsetek odrzuconych nagrań treningowych dla kilku progów
    sweep = pd.DataFrame({
        f"próg {t}": (~select_clean(train_tags, CLEAN_SELECTION["labels"], t)["clean"])
        .groupby(train_tags.index.map(source_of)).mean().mul(100)
        for t in (0.1, 0.2, 0.3, 0.5)
    }).rename_axis("zbiór")
    display(sweep.round(1).style.format("{:.1f}%"))

    selection = select_clean(train_tags, CLEAN_SELECTION["labels"], CLEAN_SELECTION["threshold"])
    clean_paths = set(selection.index[selection["clean"]])

    # Udział nagrań z objawami przed i po selekcji
    before = pd.DataFrame([vars(r) for r in train_recs])
    after = before[before["path"].astype(str).isin(clean_paths)]
    balance = pd.DataFrame({
        "nagrania przed": before.groupby("source").size(),
        "nagrania po": after.groupby("source").size(),
        "trzeszczenia przed [%]": before.groupby("source")["crackles"].mean() * 100,
        "trzeszczenia po [%]": after.groupby("source")["crackles"].mean() * 100,
        "świsty przed [%]": before.groupby("source")["wheezes"].mean() * 100,
        "świsty po [%]": after.groupby("source")["wheezes"].mean() * 100,
    }).rename_axis("zbiór")
    display(balance.round(1))

    excluded = len(train_recs) - len(after)
    if after.empty:
        raise ValueError(
            "Selekcja odrzuciła wszystkie nagrania treningowe – sprawdź próg "
            "i listę etykiet w CLEAN_SELECTION."
        )
    train_recs = [r for r in train_recs if str(r.path) in clean_paths]
    SELECTION_SUMMARY = {
        "labels": list(CLEAN_SELECTION["labels"]),
        "threshold": float(CLEAN_SELECTION["threshold"]),
        "excluded": int(excluded),
        "kept": len(train_recs),
    }
    print(f"Odrzucono {excluded} z {len(before)} nagrań treningowych "
          f"({excluded / len(before):.1%}); walidacja bez zmian ({len(val_recs)} nagrań).")

Odsłuch nagrań odrzuconych z najwyższym prawdopodobieństwem wykluczającej etykiety oraz kilku nagrań na granicy progu – pozwala ocenić, czy tagger trafnie rozpoznaje zakłócenia (pierwsze 10 s nagrania, 32 kHz).

In [ ]:
if CLEAN_SELECTION["enabled"]:
    rejected = selection[~selection["clean"]].sort_values("max_excluded_probability", ascending=False)
    borderline = selection.iloc[
        (selection["max_excluded_probability"] - CLEAN_SELECTION["threshold"]).abs().argsort()[:3]
    ]
    for title, frame in (("Najpewniej odrzucone", rejected.head(3)), ("Na granicy progu", borderline)):
        print(f"--- {title}")
        for wav_path, row in frame.iterrows():
            decision = "zachowane" if row["clean"] else "odrzucone"
            print(f"{source_of[wav_path]}: {Path(wav_path).name} – {decision}; "
                  f"{row['reason'] or f'maks. {row.max_excluded_probability:.2f}'}")
            display(Audio(load_tagger_audio(wav_path)[: 10 * 32000], rate=32000))

### 4.3. Pary czysty / zaszumiony sygnał

`NoisyCleanDataset` dzieli nagrania na segmenty 4 s (pomijając segmenty niemal ciche) i dla każdego segmentu zwraca parę `(czysty, zaszumiony)` w częstotliwości próbkowania modelu:

* **trening** – przy każdym odczycie losowane są: przesunięcie segmentu w czasie, próbka szumu i SNR z `TRAIN_SNRS_DB`,
* **walidacja** – wszystkie losowania są ustalone ziarnem, a szum pochodzi z innego generatora, więc wyniki z kolejnych epok (i różnych modeli) są porównywalne.

**Prawdziwe zakłócenia (MUSAN).** Szum syntetyczny tylko przybliża zakłócenia z nagrań – mowa jest w nim imitowana gwarem i tonem harmonicznym. Przy `MUSAN["enabled"]` część próbek szumu (`MUSAN["weight"]`) pochodzi z korpusu MUSAN: mowa (Librivox, nagrania rządu USA), muzyka i szumy techniczne / otoczenia, na licencjach Creative Commons lub w domenie publicznej:

* pliki są dzielone na treningowe i walidacyjne (rozłączne pliki; Librivox nie podaje lektorów, więc ten sam lektor może wystąpić w obu częściach), a liczba godzin na kategorię jest ograniczona (`max_hours`),
* nagrania są raz przeliczane do 4 kHz i zapisywane w `data/cache/musan_4k/` (float16, dla domyślnych limitów ok. 0,7 GB),
* każdy wycinek przechodzi przez losowy filtr dolnoprzepustowy 500–1500 Hz – przybliżenie tłumienia dźwięku z powietrza przez głowicę stetoskopu (mowa nagrana mikrofonem jest dużo jaśniejsza niż słyszana przez stetoskop) – oraz filtr górnoprzepustowy 20 Hz, jak szum syntetyczny,
* wycinki dużo cichsze niż reszta pliku (pauzy w mowie) są losowane ponownie, bo normalizacja zamieniłaby szum tła w głośny szum,
* przy włączonej selekcji CNN14 szumy MUSAN rozpoznane jako oddech, chrapanie lub świst (`RESPIRATORY_LABELS`) są pomijane – dodałyby do „szumu” to, co jest sygnałem.

**Dwie walidacje.** Walidacja syntetyczna jest taka sama jak wcześniej i to według niej wybierany jest `best.pt`, więc wyniki pozostają porównywalne z przebiegami bez MUSAN. Walidacja *prawdziwy szum* używa tych samych segmentów i poziomów SNR, ale szumu z plików walidacyjnych MUSAN – pokazuje, czy model radzi sobie z zakłóceniami spoza generatora.

**Ryzyko:** mowa jest dźwiękiem harmonicznym w paśmie zbliżonym do świstów i furczeń, więc model uczony na mowie może silniej tłumić te objawy – warto to sprawdzić w sekcji 7.3.

In [ ]:
if NOISE_DIR is None:
    base_train_noise = SyntheticClinicalNoise(PIPELINE_SAMPLE_RATE, split="train")
    val_noise = SyntheticClinicalNoise(PIPELINE_SAMPLE_RATE, split="test")
else:
    train_files, val_files = FileNoiseBank.split_files(Path(NOISE_DIR))
    base_train_noise = FileNoiseBank(train_files, PIPELINE_SAMPLE_RATE)
    val_noise = FileNoiseBank(val_files, PIPELINE_SAMPLE_RATE)

train_noise, real_val_noise, MUSAN_SUMMARY = base_train_noise, None, None
if MUSAN["enabled"]:
    musan_files = list_musan(MUSAN["root"])
    respiratory = []
    if MUSAN["exclude_respiratory"] and CLEAN_SELECTION["enabled"]:
        noise_files = musan_files[musan_files["category"] == "noise"]
        musan_tags = tag_recordings(
            noise_files["path"], TAGS_CACHE, weights=CLEAN_SELECTION["weights"],
            labels=CLEAN_SELECTION["labels_csv"], device=str(DEVICE),
        )
        respiratory_check = select_clean(musan_tags, RESPIRATORY_LABELS, CLEAN_SELECTION["threshold"])
        respiratory = respiratory_check.index[~respiratory_check["clean"]].tolist()
        print(f"Pominięte szumy MUSAN z dźwiękami oddechowymi: {len(respiratory)}")
    elif MUSAN["exclude_respiratory"]:
        print("Selekcja CNN14 wyłączona – szumy MUSAN nie są sprawdzane pod kątem oddechu.")

    musan_train, musan_val = split_musan(
        musan_files, max_hours=MUSAN["max_hours"], exclude=respiratory, seed=TRAIN["seed"]
    )
    build_musan_cache(pd.concat([musan_train, musan_val]), MUSAN_CACHE)
    musan_options = {
        "cache_dir": MUSAN_CACHE,
        "category_weights": MUSAN["category_weights"],
        "coupling_cutoff_hz": MUSAN["coupling_cutoff_hz"],
    }
    train_noise = MixedNoise({
        "syntetyczny": (base_train_noise, 1 - MUSAN["weight"]),
        "MUSAN": (MusanNoise(musan_train, **musan_options), MUSAN["weight"]),
    })
    real_val_noise = MusanNoise(musan_val, **musan_options)

    musan_table = pd.concat({"trening": musan_train, "walidacja": musan_val}, names=["część"])
    display(
        musan_table.groupby(["category", "część"])["duration"]
        .agg(pliki="size", godziny=lambda d: d.sum() / 3600)
        .unstack("część").round(2).rename_axis("kategoria")
    )
    MUSAN_SUMMARY = {
        "weight": float(MUSAN["weight"]),
        "category_weights": {k: float(v) for k, v in MUSAN["category_weights"].items()},
        "train_hours": round(float(musan_train["duration"].sum() / 3600), 2),
        "coupling_cutoff_hz": list(MUSAN["coupling_cutoff_hz"] or []),
        "excluded_respiratory": len(respiratory),
    }

SR = MODEL_OPTIONS[MODEL]["model_sample_rate"]
common = {"cache_dir": CACHE_DIR, "segment_seconds": DATA["segment_seconds"], "model_sample_rate": SR}
train_set = NoisyCleanDataset(train_recs, noise=train_noise, snrs_db=TRAIN_SNRS_DB, train=True, **common)
validation = {"recordings": val_recs, "snrs_db": TEST_SNRS_DB, "train": False,
              "seed": TRAIN["seed"], "max_segments": DATA["val_segments"], **common}
val_set = NoisyCleanDataset(noise=val_noise, **validation)
# same segments and SNRs as the synthetic validation, real noise from held-out MUSAN files
val_real_set = NoisyCleanDataset(noise=real_val_noise, **validation) if real_val_noise else None
print(f"Segmenty: {len(train_set)} treningowych, {len(val_set)} walidacyjnych (fs = {SR} Hz)"
      + ("; walidacja z prawdziwym szumem: te same segmenty" if val_real_set else ""))

Przykładowa para z walidacji. Spektrogramy pokazują, które pasma zajmuje dodany szum.

In [ ]:
def plot_spectrograms(signals: dict[str, np.ndarray], sr: int = SR, title: str | None = None) -> None:
    fig, axes = plt.subplots(2, len(signals), figsize=(4.5 * len(signals), 5), sharex=True, squeeze=False)
    for column, (name, signal) in enumerate(signals.items()):
        time_axis = np.arange(len(signal)) / sr
        limit = 1.05 * np.abs(signal).max()
        axes[0, column].plot(time_axis, signal, linewidth=0.5)
        axes[0, column].set(title=name, ylim=(-limit, limit))
        n_fft = 512 * max(1, sr // 4000)
        spec = librosa.amplitude_to_db(np.abs(librosa.stft(signal, n_fft=n_fft, hop_length=n_fft // 4)), ref=np.max)
        librosa.display.specshow(spec, sr=sr, hop_length=n_fft // 4, x_axis="time", y_axis="hz",
                                 ax=axes[1, column], vmin=-80, vmax=0)
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()


clean, noisy = (x.numpy() for x in val_set[0])
input_snr = 10 * np.log10(np.sum(clean**2) / np.sum((noisy - clean) ** 2))
plot_spectrograms({"czysty": clean, "zaszumiony": noisy}, title=f"SNR wejściowy: {input_snr:.1f} dB")
display(Audio(clean, rate=SR), Audio(noisy, rate=SR))
if val_real_set is not None:
    _, noisy_real = (x.numpy() for x in val_real_set[0])
    plot_spectrograms({"czysty": clean, "prawdziwy szum (MUSAN)": noisy_real},
                      title="Ten sam segment z szumem z MUSAN")
    display(Audio(noisy_real, rate=SR))

---

## 5. Model i przepis treningowy

Model to zwykła sieć PyTorch (`nn.Module`) z konfiguracją w `model.config` (dataclass, np. `WaveUNetConfig`) i metodą `enhance`, która zawiera normalizację potrzebną danemu modelowi – trening i późniejsze odszumianie przetwarzają więc sygnał w ten sam sposób. Checkpoint przechowuje nazwę modelu, konfigurację i wagi, dzięki czemu `load_model` odtwarza dokładnie tę samą sieć.

Przepis treningowy określa wszystko, czym trening różni się między modelami:

| Model | Funkcja straty | Optymalizacja | Źródło |
|---|---|---|---|
| Wave-U-Net | L1 na fali po normalizacji szczytu (opcjonalnie + MR-STFT) | Adam, stałe lr 1e-4, batch 16 | Stoller i in. (2018) |
| MANNER | L1 + wielorozdzielcza strata STFT (0,5/0,5) liczona dla sygnału i dla usuniętego szumu, ważona udziałem energii sygnału | Adam + OneCycleLR (maks. lr 1e-3), batch 4 | oficjalny kod MANNER |
| CMGAN | 0,1·RI + 0,9·amplituda + 0,2·fala + 0,05·GAN; dyskryminator metryczny przewiduje SI-SDR (zamiast PESQ, który dotyczy mowy) | AdamW lr 5e-4, spadek lr o połowę co 1/4 treningu, batch 4, wycinki 2 s | oficjalny kod CMGAN |

MANNER był oryginalnie trenowany przez 350 epok na VoiceBank+DEMAND; domyślne 30 epok odpowiada budżetowi treningu od zera u Tzeng i in. Rozdzielczości straty STFT są zdefiniowane dla 16 kHz i skalowane proporcjonalnie do częstotliwości próbkowania modelu, aby obejmowały te same odcinki czasu.

In [ ]:
model = build_model(MODEL, **MODEL_OPTIONS[MODEL])
if INIT_CHECKPOINT is not None and not TRAIN["resume"]:
    load_weights(model, INIT_CHECKPOINT)  # błąd, jeśli architektura się nie zgadza
    print(f"Wagi startowe: {INIT_CHECKPOINT}")
model = model.to(DEVICE)

recipe = recipe_for(model, **RECIPE_OPTIONS.get(MODEL, {})).to(DEVICE)
for key, value in recipe.defaults().items():
    if TRAIN[key] is None:
        TRAIN[key] = value

loader_options = {
    "batch_size": TRAIN["batch_size"],
    "num_workers": TRAIN["num_workers"],
    "pin_memory": DEVICE.type == "cuda",
    "persistent_workers": TRAIN["num_workers"] > 0,
}
train_loader = DataLoader(train_set, shuffle=True, drop_last=True, **loader_options)
val_loader = DataLoader(val_set, shuffle=False, **loader_options)
val_real_loader = DataLoader(val_real_set, shuffle=False, **loader_options) if val_real_set else None
steps_per_epoch = len(train_loader) if TRAIN["max_steps"] is None else min(len(train_loader), TRAIN["max_steps"])

optimizer = recipe.optimizer(model.parameters(), TRAIN["lr"])
scheduler = recipe.scheduler(optimizer, steps_per_epoch, TRAIN["epochs"], TRAIN["lr"])

n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {n_params / 1e6:.2f} M parametrów")
print("Konfiguracja sieci:", model_config(model))
print("Przepis:", recipe.source)
print(f"Trening: {TRAIN['epochs']} epok × {steps_per_epoch} kroków, batch {TRAIN['batch_size']}, "
      f"lr {TRAIN['lr']:g}, harmonogram: {type(scheduler).__name__ if scheduler else 'brak'}")

---

## 6. Trening

Jeden krok treningu wywołuje `recipe.loss`, który przepuszcza zaszumiony sygnał przez sieć dokładnie tak, jak przy odszumianiu, i zwraca stratę oraz sygnał odszumiony w oryginalnej skali amplitudy.

Po każdej epoce liczony jest średni **SI-SDR** (scale-invariant signal-to-distortion ratio, w dB) na walidacji dla sygnału zaszumionego i odszumionego. SI-SDR nie zależy od funkcji straty, więc pozwala porównywać modele trenowane różnymi przepisami. `best.pt` to epoka z najwyższym SI-SDR po odszumieniu, a `last.pt` służy do wznawiania treningu.

In [ ]:
def autocast():
    if TRAIN["amp"] and DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


def train_epoch(epoch):
    model.train()
    losses = []
    batches = iter(train_loader)
    progress = tqdm(range(steps_per_epoch), desc=f"epoka {epoch}", leave=False)
    for _ in progress:
        clean, noisy = (x.to(DEVICE, non_blocking=True) for x in next(batches))
        loss, _ = recipe.loss(model, clean, noisy, autocast)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if TRAIN["grad_clip"] > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN["grad_clip"])
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        losses.append(loss.item())
        progress.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")
    return float(np.mean(losses))


@torch.inference_mode()
def evaluate(net, loader):
    """Strata oraz SNR wejściowy i SI-SDR przed/po odszumieniu dla każdego segmentu."""
    net.eval()
    losses, rows = [], []
    for clean, noisy in tqdm(loader, desc="walidacja", leave=False):
        clean, noisy = clean.to(DEVICE), noisy.to(DEVICE)
        loss, enhanced = recipe.loss(net, clean, noisy, autocast)
        losses.append(loss.item())
        snr_in = 10 * torch.log10(clean.pow(2).sum(-1) / (noisy - clean).pow(2).sum(-1).clamp_min(1e-12))
        rows.extend(
            {"snr_in": snr, "si_sdr_noisy": before, "si_sdr_enhanced": after}
            for snr, before, after in zip(
                snr_in.tolist(), si_sdr(noisy, clean).tolist(), si_sdr(enhanced, clean).tolist(),
                strict=True,
            )
        )
    return float(np.mean(losses)), pd.DataFrame(rows)


def save_checkpoint(path, epoch, best_si_sdr, stale_epochs, real_si_sdr=None):
    torch.save(
        {
            "model": MODEL,
            "state_dict": model.state_dict(),
            "config": model_config(model),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "epoch": int(epoch),
            "best_val_si_sdr": float(best_si_sdr),
            "epochs_without_improvement": int(stale_epochs),
            "train_args": {key: str(value) for key, value in {**TRAIN, **DATA}.items()},
            "init_checkpoint": str(INIT_CHECKPOINT) if INIT_CHECKPOINT else None,
            "noise_version": NOISE_VERSION if NOISE_DIR is None else f"files:{NOISE_DIR}",
            "clean_selection": SELECTION_SUMMARY,
            "musan": MUSAN_SUMMARY,
            "val_real_si_sdr": real_si_sdr,  # SI-SDR na prawdziwym szumie w tej epoce
        },
        path,
    )

In [ ]:
history_path = OUT_DIR / "training_log.csv"
start_epoch, best_si_sdr, stale_epochs = 1, -float("inf"), 0
history = []

if TRAIN["resume"] and (OUT_DIR / "last.pt").exists():
    state = load_checkpoint(OUT_DIR / "last.pt")
    if state["config"] != model_config(model):
        raise ValueError(f"Konfiguracja sieci różni się od zapisanej w last.pt: {state['config']}")
    model.load_state_dict(state["state_dict"])
    optimizer.load_state_dict(state["optimizer"])
    if scheduler is not None and state.get("scheduler"):
        scheduler.load_state_dict(state["scheduler"])
    start_epoch = int(state["epoch"]) + 1
    best_si_sdr = float(state["best_val_si_sdr"])
    stale_epochs = int(state.get("epochs_without_improvement", 0))
    if history_path.exists():
        history = pd.read_csv(history_path).to_dict("records")
    print(f"Wznowiono od epoki {start_epoch} (najlepszy SI-SDR: {best_si_sdr:.2f} dB)")

for epoch in range(start_epoch, TRAIN["epochs"] + 1):
    started = time.perf_counter()
    train_loss = train_epoch(epoch)
    val_loss, val_scores = evaluate(model, val_loader)
    noisy_score = float(val_scores["si_sdr_noisy"].mean())
    enhanced_score = float(val_scores["si_sdr_enhanced"].mean())
    real_noisy = real_enhanced = float("nan")
    if val_real_loader is not None:
        _, real_scores = evaluate(model, val_real_loader)
        real_noisy = float(real_scores["si_sdr_noisy"].mean())
        real_enhanced = float(real_scores["si_sdr_enhanced"].mean())

    improved = enhanced_score > best_si_sdr
    best_si_sdr = max(best_si_sdr, enhanced_score)
    stale_epochs = 0 if improved else stale_epochs + 1
    real_for_checkpoint = None if np.isnan(real_enhanced) else real_enhanced
    save_checkpoint(OUT_DIR / "last.pt", epoch, best_si_sdr, stale_epochs, real_for_checkpoint)
    if improved:
        save_checkpoint(OUT_DIR / "best.pt", epoch, best_si_sdr, stale_epochs, real_for_checkpoint)

    history.append({
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_si_sdr_noisy": noisy_score,
        "val_si_sdr_enhanced": enhanced_score,
        "val_real_si_sdr_noisy": real_noisy,
        "val_real_si_sdr_enhanced": real_enhanced,
        "seconds": time.perf_counter() - started,
    })
    pd.DataFrame(history).to_csv(history_path, index=False)
    print(f"epoka {epoch:2d} | trening {train_loss:.4f} | walidacja {val_loss:.4f} | "
          f"SI-SDR {noisy_score:5.2f} → {enhanced_score:5.2f} dB"
          + (f" | prawdziwy szum {real_noisy:5.2f} → {real_enhanced:5.2f} dB"
             if not np.isnan(real_enhanced) else "")
          +
          f"{'  (najlepsza)' if improved else ''} | {history[-1]['seconds']:.0f} s")

    if TRAIN["patience"] and stale_epochs >= TRAIN["patience"]:
        print(f"Early stopping: {TRAIN['patience']} epok bez poprawy.")
        break

---

## 7. Ocena

### 7.1. Krzywe uczenia

Strata nie jest porównywalna między modelami (różne przepisy), SI-SDR – tak.

In [ ]:
history_df = pd.read_csv(history_path)

fig, (ax_loss, ax_sdr, ax_lr) = plt.subplots(1, 3, figsize=(15, 4))
ax_loss.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="trening")
ax_loss.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="walidacja")
ax_loss.set(xlabel="epoka", ylabel="strata", title=f"Strata ({MODEL})")
ax_loss.legend()

ax_sdr.plot(history_df["epoch"], history_df["val_si_sdr_enhanced"], marker="o", label="odszumiony (syntetyczny)")
ax_sdr.axhline(history_df["val_si_sdr_noisy"].iloc[0], color="gray", linestyle="--", label="zaszumiony (syntetyczny)")
if history_df.get("val_real_si_sdr_enhanced", pd.Series(dtype=float)).notna().any():
    real = history_df.dropna(subset=["val_real_si_sdr_enhanced"])
    ax_sdr.plot(real["epoch"], real["val_real_si_sdr_enhanced"], marker="s", label="odszumiony (MUSAN)")
    ax_sdr.axhline(real["val_real_si_sdr_noisy"].iloc[0], color="gray", linestyle=":", label="zaszumiony (MUSAN)")
ax_sdr.set(xlabel="epoka", ylabel="SI-SDR [dB]", title="SI-SDR na walidacji")
ax_sdr.legend()

ax_lr.plot(history_df["epoch"], history_df["lr"], marker="o")
ax_lr.set(xlabel="epoka", ylabel="lr", title="Współczynnik uczenia (koniec epoki)", yscale="log")
fig.tight_layout()
plt.show()

display(history_df.style.format({"lr": "{:.1e}", "seconds": "{:.0f}"}, precision=3).hide(axis="index"))

### 7.2. Jakość według poziomu szumu

Najlepszy checkpoint jest wczytywany przez `load_model`, tak samo jak przy odszumianiu zbioru (sekcja 9). Poprawa SI-SDR jest zwykle największa dla najsilniejszego szumu (najniższy SNR). Przy słabym szumie model może pogarszać sygnał, bo zmienia także to, co było czyste – to ważna informacja przed użyciem modelu na zwykłych nagraniach.

In [ ]:
best = load_model(OUT_DIR / "best.pt", device=DEVICE)
print(f"Najlepsza epoka: {load_checkpoint(OUT_DIR / 'best.pt')['epoch']}")


def table_by_snr(scores):
    scores = scores.assign(
        snr_level=scores["snr_in"].apply(lambda v: min(TEST_SNRS_DB, key=lambda level: abs(level - v))),
        poprawa=scores["si_sdr_enhanced"] - scores["si_sdr_noisy"],
    )
    columns = ["si_sdr_noisy", "si_sdr_enhanced", "poprawa"]
    table = scores.groupby("snr_level")[columns].mean()
    table["segmenty"] = scores.groupby("snr_level").size()
    table.loc["wszystkie"] = [*scores[columns].mean(), len(scores)]  # średnia po wszystkich segmentach
    return table.rename_axis("SNR wejściowy [dB]").rename(
        columns={"si_sdr_noisy": "SI-SDR zaszumiony", "si_sdr_enhanced": "SI-SDR odszumiony"}
    ).round(2).astype({"segmenty": int})


_, scores = evaluate(best, val_loader)
print("Szum syntetyczny:")
display(table_by_snr(scores))
if val_real_loader is not None:
    _, real_scores = evaluate(best, val_real_loader)
    print("Prawdziwy szum (MUSAN, pliki walidacyjne):")
    display(table_by_snr(real_scores))

### 7.3. Przykłady odszumiania

Najpierw segment walidacyjny ze sztucznym szumem, dla którego znany jest czysty wzorzec. Następnie całe **prawdziwe nagranie** z walidacji, bez dodanego szumu, przetworzone dokładnie tak jak w pipeline: wczytanie w 4 kHz, podział na segmenty z zakładką, odszumienie, sklejenie. Na końcu nagrania odrzucone przez selekcję z sekcji 4.2, czyli przykłady prawdziwych zakłóceń.

In [ ]:
example = int(scores["snr_in"].idxmin())  # segment z najsilniejszym szumem
clean, noisy = (x.numpy() for x in val_set[example])
with torch.inference_mode():
    _, enhanced = recipe.loss(best, torch.from_numpy(clean)[None].to(DEVICE),
                              torch.from_numpy(noisy)[None].to(DEVICE), autocast)
enhanced = enhanced[0].float().cpu().numpy()

plot_spectrograms(
    {"czysty": clean, "zaszumiony": noisy, "odszumiony": enhanced},
    title=f"SNR wejściowy {scores.loc[example, 'snr_in']:.1f} dB",
)
display(Audio(clean, rate=SR), Audio(noisy, rate=SR), Audio(enhanced, rate=SR))

In [ ]:
recording = val_recs[0]
audio, sr = _load_feature_audio(str(recording.path), PIPELINE_SAMPLE_RATE)
denoised = enhance_signal(best, audio)

print(f"{recording.source}: {recording.path.name}, {len(audio) / sr:.1f} s")
plot_spectrograms({"oryginał": audio, "odszumiony": denoised, "usunięte": audio - denoised},
                  sr=sr, title="Prawdziwe nagranie")
display(Audio(audio, rate=sr), Audio(denoised, rate=sr), Audio(audio - denoised, rate=sr))

**Nagrania odrzucone przez selekcję.** Tagger CNN14 rozpoznał w nich prawdziwe zakłócenia (np. mowę, muzykę), więc nie zostały użyte w treningu – to test na zakłóceniach, których model nie widział i które nie pochodzą z generatora szumu. Dla porównania model przetwarza też 3 nagrania uznane za najczystsze (najniższe prawdopodobieństwo wykluczających etykiet). Przetwarzane jest pierwsze 20 s każdego nagrania, tak jak w pipeline (4 kHz, segmenty z zakładką).

Tabela podaje, jaką część energii nagrania model usunął. Jeżeli radzi sobie z prawdziwymi zakłóceniami, z nagrań odrzuconych powinien usuwać wyraźnie więcej niż z czystych. Mała różnica oznacza, że model usuwa głównie szum podobny do syntetycznego – był trenowany tylko na nim. Warto też posłuchać części „usunięte”: jeżeli słychać w niej oddech, świsty lub trzeszczenia, model usuwa także sygnał.

In [ ]:
if CLEAN_SELECTION["enabled"] and SELECTION_SUMMARY is not None:
    rejected = selection[~selection["clean"]].sort_values("max_excluded_probability", ascending=False)
    cleanest = selection[selection["clean"]].sort_values("max_excluded_probability")
    examples = [("odrzucone", path, row) for path, row in rejected.head(3).iterrows()]
    examples += [("zachowane", path, row) for path, row in cleanest.head(3).iterrows()]

    removal_rows = []
    for decision, wav_path, row in examples:
        audio, sr = _load_feature_audio(wav_path, PIPELINE_SAMPLE_RATE)
        audio = audio[: 20 * sr]
        denoised = enhance_signal(best, audio)
        removed = audio - denoised
        removed_share = float(np.sum(removed**2) / max(float(np.sum(audio**2)), 1e-12))
        reason = row["reason"] or f"maks. {row['max_excluded_probability']:.2f}"
        removal_rows.append({
            "nagranie": Path(wav_path).name,
            "zbiór": source_of[wav_path],
            "decyzja": decision,
            "etykiety CNN14": reason,
            "usunięta energia [%]": 100 * removed_share,
        })
        if decision == "odrzucone":
            plot_spectrograms(
                {"oryginał": audio, "odszumiony": denoised, "usunięte": removed},
                sr=sr,
                title=f"{source_of[wav_path]}: {Path(wav_path).name} – {reason}",
            )
            display(Audio(audio, rate=sr), Audio(denoised, rate=sr), Audio(removed, rate=sr))

    display(pd.DataFrame(removal_rows).set_index("nagranie").round(1))
else:
    print("Selekcja wyłączona – brak nagrań odrzuconych do porównania.")

---

## 8. Porównanie modeli

Zestawienie wszystkich wytrenowanych modeli z folderu `models/` (każdy podfolder z `best.pt` i `training_log.csv`). Wszystkie przebiegi z tym samym ziarnem i tymi samymi danymi walidacyjnymi mają ten sam SI-SDR sygnału zaszumionego, więc wyniki są bezpośrednio porównywalne. Przebiegi testowe (`*_smoke`) są pomijane.

**Porównywać można tylko wiersze z tą samą wersją szumu.** Kolumna *selekcja* pokazuje, czy nagrania treningowe były filtrowane taggerem CNN14 (próg i liczba odrzuconych nagrań); walidacja jest zawsze ta sama, więc SI-SDR różnych wariantów selekcji jest porównywalny. Kolumny *prawdziwy szum* (walidacja z szumem z MUSAN, dla epoki `best.pt`) są wypełnione tylko dla przebiegów z walidacją MUSAN – można je porównywać między przebiegami z tym samym podziałem MUSAN. Wersja 1 (przebiegi sprzed 22.11.2026) miała dwa błędy generatora: większość mocy szumu „otoczenia” leżała poniżej 20 Hz, a zgrzyty tarcia były ok. 60× za głośne względem ciągłego pocierania. Wersja 2 je poprawia, więc daje inne dane treningowe i walidacyjne.

In [ ]:
rows = []
for run_dir in sorted((ROOT / "models").iterdir()):
    if not (run_dir / "best.pt").exists() or not (run_dir / "training_log.csv").exists():
        continue
    if run_dir.name.endswith("_smoke") and not SMOKE_TEST:
        continue
    state = load_checkpoint(run_dir / "best.pt")
    name = state.get("model")
    if name not in MODELS:
        continue
    log = pd.read_csv(run_dir / "training_log.csv")
    network = build_model(name, **state["config"])
    rows.append({
        "przebieg": run_dir.name,
        "model": name,
        "wersja szumu": state.get("noise_version", 1),  # brak wpisu = v1
        "selekcja": (
            f"CNN14 ≥ {sel['threshold']} (−{sel['excluded']})"
            if (sel := state.get("clean_selection")) else "brak"
        ),
        "parametry [M]": sum(p.numel() for p in network.parameters()) / 1e6,
        "fs modelu [Hz]": state["config"].get("model_sample_rate"),
        "epoki": len(log),
        "najlepsza epoka": state["epoch"],
        "SI-SDR zaszumiony": log["val_si_sdr_noisy"].iloc[-1],
        "SI-SDR odszumiony": state["best_val_si_sdr"],
        "poprawa [dB]": state["best_val_si_sdr"] - log["val_si_sdr_noisy"].iloc[-1],
        "MUSAN": f"{m['weight']:.0%}" if (m := state.get("musan")) else "—",
        "prawdziwy szum: zaszumiony": (
            log["val_real_si_sdr_noisy"].dropna().iloc[-1]
            if "val_real_si_sdr_noisy" in log and log["val_real_si_sdr_noisy"].notna().any() else np.nan
        ),
        "prawdziwy szum: odszumiony": state.get("val_real_si_sdr") or np.nan,
        "czas epoki [s]": log["seconds"].median(),
    })

comparison = pd.DataFrame(rows).set_index("przebieg") if rows else pd.DataFrame()
display(comparison.round(2))

---

## 9. Użycie wytrenowanego modelu

Odszumianie jest osobnym krokiem: każde nagranie jest odszumiane raz, a wynik zapisywany do folderu (4 kHz, `.npy`). Pipeline czyta potem ten folder zamiast oryginalnych plików `.wav`:

```python
from src.denoising import enhance_signal, load_model, save_processed_audio

model = load_model("../models/wave_u_net/best.pt", device="cuda")
audio_dir = Path("../data/denoised/wave_u_net")
save_processed_audio(wav_paths, audio_dir, lambda audio: enhance_signal(model, audio))

df = pipeline.build_dataset_with_features(audio_dir=audio_dir)
```

`enhance_signal` dzieli nagranie na 4-sekundowe segmenty z zakładką 0,5 s (długość, na której model był trenowany), odszumia je wsadowo i skleja z płynnym przejściem. `save_processed_audio` pomija pliki, które już istnieją – **po ponownym treningu modelu trzeba użyć nowego folderu** (albo usunąć stary). Klucz cache cech w pipeline uwzględnia listę plików w `audio_dir` wraz z datami modyfikacji, więc nowe audio oznacza nowe cechy.